# Olist Dataset — Schema Understanding

## Objective

The purpose of this notebook is to understand the structural organization of the Olist dataset before designing transformation pipelines and dimensional models.

This analysis focuses on:
- Dataset inventory
- Schema inspection
  * Candidate primary key
  * Candidate foreign keys
- Data granularity
- Table relationships

#### *Import Libraries*

The following libraries are used for:
- loading tabular datasets
- handling file paths
- performing schema inspection and profiling

In [2]:
import pandas as pd
from pathlib import Path

#### *Define Data Source Path*

A centralized base path is defined to avoid hardcoded file locations and improve maintainability across the project structure.

In [ ]:
DATA_PATH = Path(''../data/raw')

#### *Load Raw Datasets*

Datasets are loaded into a dictionary structure to centralize access and simplify future validation, iteration, and automation tasks.

In [4]:
datasets = {
    "customers": pd.read_csv(DATA_PATH / "olist_customers_dataset.csv"),
    "geolocation": pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv"),
    "order_items": pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv"),
    "order_payments": pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv"),
    "order_reviews": pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv"),
    "orders": pd.read_csv(DATA_PATH / "olist_orders_dataset.csv"),
    "products": pd.read_csv(DATA_PATH / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv"),
    "product_category_translation": pd.read_csv(DATA_PATH / "product_category_name_translation.csv")
}

#### *Validate Dataset Loading*

Initial validation is performed to confirm that all datasets were loaded successfully and to inspect their dimensions.

In [5]:
datasets.keys()

dict_keys(['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_translation'])

### Dataset Inventory

First view about:
- Tables
- Number of rows
- Number of columns
- Sizes

In [ ]:
sumarized_list  = []
total_size_mb = 0

for name, df in datasets.items():
    # calculate memory used to tables in MB
    size_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    total_size_mb += size_mb

    sumarized_list.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Size (MB)': round(size_mb, 2)
    })

df_sumarized = pd.DataFrame(sumarized_list)
# display DataFrame without index
display(df_sumarized.style.hide(axis='index'))

# calculate memory used to all dataset in GB
total_size_gb = total_size_mb / 1024
print('-' * 50)
print(f'Tamaño Total: {total_size_gb:.2f} GB')

Dataset,Rows,Columns,Size (MB)
customers,99441,5,26.590000
geolocation,1000163,5,129.380000
order_items,112650,7,35.990000
order_payments,103886,5,16.230000
order_reviews,99224,7,39.120000
orders,99441,8,52.940000
products,32951,9,6.300000
sellers,3095,4,0.590000
product_category_translation,71,2,0.010000


--------------------------------------------------
Tamaño Total: 0.30 GB


### Initial Schema Inspection - Orders Table

The orders table represents the central transactional entity of the marketplace

This inspection aims to:
- understand the table structure
- identify important columns
- inspect data types
- detect potential identifiers and timestamps

#### Orders Table:

The Order Entity has 8 columns and 99441 records describing below:
* `order_id`
  * 99441 Records
  * not nulls
  * data type object -> VARCHAR - ALPHANUMERIC
* `customer_id`
  * 99441 Records
  * not nulls
  * data type object -> VARCHAR - ALPHANUMERIC
* `order_status`
  * 99441 Records
  * not nulls
  * data type object -> TEXT - VARCHAR
* `order_purchase_timestamp`
  * 99441 Records
  * not nulls
  * data type object -> DATE - TIMESTAMP
* `order_approved_at`
  * 99281 Records
  * nulls found
  * data type object -> DATE - TIMESTAMP
* `order_delivered_carrier_date`
  * 97658 Records
  * nulls found
  * data type object -> DATE - TIMESTAMP
* `order_delivered_customer_date`
  * 96476 Records
  * nulls found
  * data type object -> DATE - TIMESTAMP
* `order_estimated_delivery_date`
  * 99441 Records
  * not nulls
  * data type object -> DATE - TIMESTAMP

In [7]:
datasets["orders"].info(memory_usage=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [22]:
columns_duplicated = []

for item in datasets['orders']:
    duplicated = datasets['orders'][item].duplicated().sum()
    columns_duplicated.append({
        'Column': item,
        'Duplicated': duplicated
    })

df_columns_duplicated = pd.DataFrame(columns_duplicated)

display(df_columns_duplicated.style.hide(axis='index'))



Column,Duplicated
order_id,0
customer_id,0
order_status,99433
order_purchase_timestamp,566
order_approved_at,8707
order_delivered_carrier_date,18422
order_delivered_customer_date,3776
order_estimated_delivery_date,98982


### Preview Orders Table

A small sample of the dataset is displayed to better understand:
- identifier formats
- timestamp representations
- potential business attributes

In [8]:
datasets["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Analyze Orders Columns

Each column is inspected to understand:
- its business meaning
- its role in the order lifecycle
- possible identifiers
- potential timestamps and status indicators

### Validate Table Granularity

The granularity of the orders table is inspected to determine whether each row represents a unique business event.

- primary key identification
- table relationships
- dimensional modeling
- aggregation logic

In [9]:
datasets['orders']['order_id'].duplicated().sum()

np.int64(0)

### Granularity Observation

The `order_id` column does not contain duplicated values, suggesting that:
- each row represents a unique order
- the table grain is likely `1 row = 1 order`
- `order_id` is a strong candidate for a primary key

In [10]:
datasets['orders']['customer_id'].duplicated().sum()


np.int64(0)

The `customer_id` column does not contain duplicated values, suggesting that:
- if `1 customer = 1 order` could be business inconsitency
- the `customer_id` is likely technical/operational identifier
- the `customer_id` is possible entity linking to customer table (FK)

In [11]:
datasets["customers"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


## Schema Inspection — Customers Table

The customers table is inspected to understand how customer entities are represented in the dataset.

This analysis focuses on:
- customer identifiers
- customer uniqueness
- potential relationships with the orders table
- customer location attributes

In [12]:
datasets["customers"]['customer_id'].duplicated().sum()

np.int64(0)

In [13]:
datasets["customers"]['customer_unique_id'].duplicated().sum()

np.int64(3345)

#### Understanding business behavior and entity representation

The `customer_id` has no duplicate records, which suggests:
- Order identifier
- It acts as the operational entity to link with the orders table

The `customer_unique_id` has records duplicated, that suggests:
- It is the actual customer identifier
- It indicates repeat customers (customer retention)
- IIt represents the core business entity

### Validating cross-table consistency between orders-customers

Data consistency is verified between both entities:
- The `customer_id` column (PK) from `orders` table is correctly linked with `customer_id` column from `customers` table, and vice versa.
- Confirming the key consitency

In [14]:
datasets['orders']['customer_id'].isin(datasets['customers']['customer_id']).all()

np.True_

In [15]:
datasets['customers']['customer_id'].isin(datasets['orders']['customer_id']).all()

np.True_